In [1]:
import json
import os
import re
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import pandas as pd
from openai import OpenAI
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

project_root = Path.cwd().parent.parent


def _normalize_text(value: object) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _normalize_phone(value: object, default_region: str = "CA") -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    raw = str(value).strip()
    if not raw:
        return ""
    digits_only = re.sub(r"\D", "", raw)
    try:
        import phonenumbers

        parsed = phonenumbers.parse(raw, default_region)
        if not phonenumbers.is_valid_number(parsed):
            return digits_only
        return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)
    except Exception:
        return digits_only


def _column_score(a: str, b: str) -> Optional[float]:
    if not a or not b:
        return None
    jw = JaroWinkler.similarity(a, b)
    ts = fuzz.token_set_ratio(a, b) / 100.0
    return max(jw, ts)


def _build_candidate_pairs(embeddings: np.ndarray, threshold: float) -> set[tuple[int, int]]:
    radius = 1.0 - threshold
    nn = NearestNeighbors(metric="cosine", radius=radius)
    nn.fit(embeddings)
    distances, indices = nn.radius_neighbors(embeddings, radius=radius)
    pairs: set[tuple[int, int]] = set()
    for i, neighbors in enumerate(indices):
        for j in neighbors:
            if i == j:
                continue
            a, b = (i, j) if i < j else (j, i)
            pairs.add((a, b))
    return pairs


def llm_judge(
    pairs: list[dict],
    model: Optional[str] = None,
    batch_size: int = 10,
) -> list[bool]:
    client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL", "gpt-5-nano")
    decisions: list[bool] = []

    system_prompt = (
        "You are a deduplication judge. Decide if two records describe the same physical facility. "
        "Return JSON only."
    )

    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        payload = [
            {
                "left": item["left"],
                "right": item["right"],
            }
            for item in batch
        ]

        user_prompt = (
            "For each pair, decide if they refer to the same physical facility. "
            "If either side has missing values, use best judgment. "
            "Reply with JSON in the form {\"decisions\":[true,false,...]} in the same order.\n" 
            f"Pairs: {json.dumps(payload, ensure_ascii=True)}"
        )

        response = client.responses.create(
            model=model,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        text = response.output_text or ""
        try:
            parsed = json.loads(text)
            batch_decisions = parsed.get("decisions", [])
        except json.JSONDecodeError:
            batch_decisions = []

        if len(batch_decisions) != len(batch):
            batch_decisions = [False] * len(batch)

        decisions.extend(bool(x) for x in batch_decisions)

    return decisions


def dedupe_csvs(
    input_csv_path: str | Path,
    output_csv_path: str | Path,
    hard_id_col: Optional[str] = "phone",
    soft_id_cols: Optional[list[str]] = None,
    default_region: str = "CA",
    vector_threshold: float = 0.80,
    auto_merge_threshold: float = 0.90,
    auto_keep_threshold: float = 0.60,
    llm_judge: Optional[Callable[[list[dict]], list[bool]]] = None,
) -> Path:
    if soft_id_cols is None:
        soft_id_cols = ["address", "name"]

    if not input_csv_path:
        raise ValueError("input_csv_path is required.")

    df = pd.read_csv(input_csv_path)
    raw_rows = len(df)

    # Normalize soft columns
    for col in soft_id_cols:
        if col not in df.columns:
            df[col] = ""
        df[f"__norm_{col}"] = df[col].apply(_normalize_text)

    # Normalize hard id column if provided
    hard_norm_col = None
    if hard_id_col and hard_id_col in df.columns:
        hard_norm_col = f"__norm_{hard_id_col}"
        df[hard_norm_col] = df[hard_id_col].apply(lambda v: _normalize_phone(v, default_region))

    # Layer 0: Hard-ID dedupe
    post_hard_rows = raw_rows
    if hard_norm_col:
        has_hard = df[hard_norm_col].astype(bool)
        hard_dupes = df[has_hard].duplicated(subset=[hard_norm_col], keep="first")
        drop_idx = df[has_hard][hard_dupes].index
        df = df.drop(index=drop_idx).reset_index(drop=True)
        post_hard_rows = len(df)

    # Layer 1: Vector blocking
    df["__soft_text"] = df[[f"__norm_{c}" for c in soft_id_cols]].agg(" ".join, axis=1).str.strip()
    non_empty_mask = df["__soft_text"].astype(bool)
    if non_empty_mask.any():
        model = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = model.encode(
            df["__soft_text"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings)
        print(f"Embeddings: {embeddings.shape}")
        candidate_pairs = _build_candidate_pairs(embeddings, vector_threshold)
    else:
        candidate_pairs = set()

    # Layer 2 and 3: Strict verifier and waterfall decision
    duplicates: set[tuple[int, int]] = set()
    gray_pairs: list[tuple[int, int]] = []

    for i, j in candidate_pairs:
        if not df.at[i, "__soft_text"] or not df.at[j, "__soft_text"]:
            gray_pairs.append((i, j))
            continue
        scores = []
        missing_data = False
        for col in soft_id_cols:
            a = df.at[i, f"__norm_{col}"]
            b = df.at[j, f"__norm_{col}"]
            score = _column_score(a, b)
            if score is None:
                missing_data = True
                break
            scores.append(score)
        if missing_data or not scores:
            gray_pairs.append((i, j))
            continue
        final_score = min(scores)
        if final_score >= auto_merge_threshold:
            duplicates.add((i, j))
        elif final_score < auto_keep_threshold:
            continue
        else:
            gray_pairs.append((i, j))

    # Layer 4: Optional LLM judge
    if gray_pairs and llm_judge:
        pair_payload = []
        for i, j in gray_pairs:
            pair_payload.append({
                "left": df.loc[i, soft_id_cols + [hard_id_col]].to_dict() if hard_id_col else df.loc[i, soft_id_cols].to_dict(),
                "right": df.loc[j, soft_id_cols + [hard_id_col]].to_dict() if hard_id_col else df.loc[j, soft_id_cols].to_dict(),
                "index_pair": (i, j),
            })
        decisions = llm_judge(pair_payload)
        for payload, is_dup in zip(pair_payload, decisions):
            if is_dup:
                duplicates.add(payload["index_pair"])

    # Union-Find to merge duplicates
    parent = list(range(len(df)))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i, j in duplicates:
        union(i, j)

    # Keep first record in each cluster
    seen_roots: set[int] = set()
    keep_indices: list[int] = []
    for idx in range(len(df)):
        root = find(idx)
        if root not in seen_roots:
            seen_roots.add(root)
            keep_indices.append(idx)

    df_deduped = df.loc[keep_indices].drop(columns=[c for c in df.columns if c.startswith("__")])
    output_csv_path = Path(output_csv_path)
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_deduped.to_csv(output_csv_path, index=False)

    print(f"Input rows (raw): {raw_rows}")
    if hard_norm_col:
        print(f"After hard-id dedupe: {post_hard_rows}")
    print(f"Deduped rows: {len(df_deduped)}")
    print(f"Saved CSV: {output_csv_path}")
    return output_csv_path

In [9]:
df = pd.read_csv("ontario_211_eval_200.csv")
df[df["duplicate_status"] == False]

,name,address,phone,duplicate_status
3,Montfort Academic Family Health Team,"745 Montreal Rd, Suite 101B Ottawa, ON, K1K 0T...",613-749-4429,False
4,Health Sciences North,"127 Cedar St Greater Sudbury, ON, P3E 1B1 (2km...","705-523-4988, 211, 705-675-4732, 705-368-0756",False
5,Playsmart Centre,"OLG Slots at Georgian Downs, 7485 5th Side Rd ...","1-866-986-9055, 705-329-1269, 705-797-2329",False
7,Peterborough Regional Health Centre,"432 George St N Peterborough, ON, K9H 3R5 Clip...",705-749-9708,False
8,Alderville First Nation,"8467 County Rd 45 Alderville First Nation, ON,...",905-352-2140,False
...,...,...,...,...
187,Men's Counselling Services,"43 King St W, Suite 204 Brockville, ON, K6V 3P...",613-498-1940,False
190,Eastern Ontario Health Unit,"1000 Pitt St Cornwall, ON, K6J 5T1 (1km) Clipb...","1-800-267-7120, 613-933-1375",False
191,Southwestern Public Health Oxford Elgin St. Th...,"410 Buller St Woodstock, ON, N4S 4N2 (1km) Cli...","1-800-922-0096, 519-421-9901",False
193,Huron Perth Public Health,"653 West Gore St Stratford, ON, N5A 1L4 (2km) ...","1-888-221-2133, 1-888-221-2133",False


In [3]:
import pandas as pd

def evaluate_deduplication(gt_file, output_file):
    """
    Evaluates deduplication performance on the Hard Mode V2 dataset.
    Uses 'group_id' to track entities even if names are twisted.
    """
    print(f"📊 Loading files...")
    # Load GT and Output
    gt = pd.read_csv(gt_file)
    output = pd.read_csv(output_file)
    
    # 1. Create a Lookup Map from the Ground Truth
    # We map "Twisted Name + Twisted Address" -> Group ID
    # This allows us to identify which entity a row belongs to.
    gt['signature'] = gt['name'].astype(str).str.strip() + "|" + gt['address'].astype(str).str.strip()
    signature_to_group = dict(zip(gt['signature'], gt['group_id']))
    
    # 2. Analyze the Output
    output['signature'] = output['name'].astype(str).str.strip() + "|" + output['address'].astype(str).str.strip()
    
    found_groups = set()
    redundant_rows = 0
    unmatched_rows = 0
    
    for sig in output['signature']:
        if sig in signature_to_group:
            group_id = signature_to_group[sig]
            
            if group_id in found_groups:
                redundant_rows += 1 # We already found this group! Duplicate!
            else:
                found_groups.add(group_id)
        else:
            # The output row doesn't match ANY row in the input? 
            # This happens if your deduper modified the text (e.g. "St." -> "Street")
            unmatched_rows += 1

    # 3. Calculate Metrics
    total_unique_groups_gt = gt['group_id'].nunique()
    
    recall = len(found_groups) / total_unique_groups_gt
    # Ideally: Total Output Rows == Total Unique Groups
    # Precision = (Unique Groups Found) / (Total Output Rows)
    precision = len(found_groups) / len(output)
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # --- REPORT ---
    print("\n--- HARD MODE EVALUATION REPORT ---")
    print(f"Total Unique Groups (GT):      {total_unique_groups_gt}")
    print(f"Groups Found in Output:        {len(found_groups)}")
    print(f"Unmatched/Modified Rows:       {unmatched_rows}")
    print("-" * 40)
    print(f"RECALL (Retention):    {recall:.2%}")
    print(f"PRECISION (Cleaning):  {precision:.2%}")
    print(f"F1 SCORE:              {f1:.4f}")
    print("-" * 40)
    
    if unmatched_rows > 0:
        print("   Warning: Some output rows couldn't be matched to the Ground Truth.")
        print("   This usually means your deduplication algo modified the text (e.g. normalization).")
        print("   To fix this evaluation, disable text normalization in your deduper output.")


In [18]:
evaluate_deduplication('ontario_211_eval_hard_mode_v2.csv', 'ontario_211_eval_hard_mode_v2_deduped.csv')

📊 Loading files...

--- HARD MODE EVALUATION REPORT ---
Total Unique Groups (GT):      110
Groups Found in Output:        110
Unmatched/Modified Rows:       0
----------------------------------------
RECALL (Retention):    100.00%
PRECISION (Cleaning):  99.10%
F1 SCORE:              0.9955
----------------------------------------


In [3]:
input_csv = "ontario_211_disability_69.csv"
output_csv = "combined_disability_69_deduped.csv"



dedupe_csvs(
    input_csv_path=input_csv,
    output_csv_path=output_csv,
    hard_id_col="phone",
    soft_id_cols=["address", "name"],
    llm_judge=llm_judge,
)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings: (297, 384)
Input rows (raw): 1391
After hard-id dedupe: 297
Deduped rows: 262
Saved CSV: combined_disability_69_deduped.csv


PosixPath('combined_disability_69_deduped.csv')

In [5]:
df1 = pd.read_csv("ontario_211_disability_17_18.csv")
df2 = pd.read_csv("ontario_211_disability_19_21.csv")
df3 = pd.read_csv("ontario_211_disability_22_24.csv")

In [6]:
df_final = pd.concat([df1, df2, df3], ignore_index=True)
df_final.to_csv("disability_data.csv", index=False)

In [10]:
df_final

,name,program,address,city,phone,website,description,source_url,topic_path,search_city
0,Tabularasa Communities,NaN,"101 Ninth St W Cornwall, ON, K6J 0A3 (3km) Cli...",Cornwall,613-890-3472,https://tabularasacommunities.ca/,Day centre offering supports for young adults ...,https://211ontario.ca/results/?latitude=45.004...,17,Akwesasne
1,Rachel's Kids,Park of Hope,"13th St W Cornwall, ON, K6J 4K5 (3km) Clipboard",Cornwall - 13th St W,NaN,https://rachelskids.com,Fully inclusive playground where children of a...,https://211ontario.ca/results/?latitude=45.004...,17,Akwesasne
2,Inspire Community Support Services,Developmental Services for Adults,"1424 Aubin Ave, Unit 1, Suite 100 Cornwall, ON...",Cornwall - Aubin Ave,613-932-4610,https://inspire-sdg.ca/,Developmental Services for Adults case managem...,https://211ontario.ca/results/?latitude=45.004...,17,Akwesasne
3,Inspire Community Support Services,Clinical & Support Services,"1424 Aubin Ave, Unit 1, Suite 100 Cornwall, ON...",Cornwall - Aubin Ave,"1-855-647-8483, 613-932-4610",https://inspire-sdg.ca,Clinical and support services General psychoth...,https://211ontario.ca/results/?latitude=45.004...,17,Akwesasne
4,Inspire Community Support Services,Developmental Services for Children - Case Man...,"1424 Aubin Ave, Unit 1, Suite 100 Cornwall, ON...",Cornwall - Aubin Ave,613-937-3072,https://inspire-sdg.ca/services/developmental-...,Case management services provides collaborativ...,https://211ontario.ca/results/?latitude=45.004...,17,Akwesasne
...,...,...,...,...,...,...,...,...,...,...
47759,DisAbled Women's Network of Canada,NaN,"469 rue Jean-Talon Ouest, Unit 215 Villeray-Sa...",Montreal,"1-866-396-0074, 514-396-0009",https://www.dawncanada.net,"Works to end the poverty, discrimination and v...",https://211ontario.ca/results/?searchLocation=...,24,Woodstock
47760,Bell Canada,TTY Relay Service,Clipboard,Ottawa - No physical location,"1-800-855-115, 7-1-1, 1-800-268-9242",https://www.bell.ca/Accessibility_services/Bel...,Teletypewriter (TTY) relay service numbers tha...,https://211ontario.ca/results/?searchLocation=...,24,Woodstock
47761,Mira Foundation,Service Dogs for Youth with Autism Spectrum Di...,"1820 Rang Nord-Ouest Sainte-Madeleine, QC, J0H...",Sainte-Madeleine,450-795-3725,https://www.mira.ca/en/,Program that provides service dogs to individu...,https://211ontario.ca/results/?searchLocation=...,24,Woodstock
47762,Geri Fashions,Adapted Clothing,Clipboard,Online Service,1-800-361-4374,https://gerifashions.com,Offers for sale online men's and women's easy-...,https://211ontario.ca/results/?searchLocation=...,24,Woodstock
